# 🚦 TrafficPulse: K-Means ML Analysis on Real Traffic Flow Dataset (`Traffic_Flow_Dataset.csv`)

**Objective**: Perform comprehensive machine learning analysis using **K-Means Clustering** on the actual sensor dataset (`Traffic_Flow_Dataset.csv`). We preprocess traffic volume, vehicle speed, density, queue length, signal delay, and weather factors, scale features with Z-score standardization, evaluate cluster count using the **Elbow Method (WCSS)**, and map congestion profiles.

---

## 1. Environment & Library Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('crest')
print('Data science libraries imported successfully.')

## 2. Load & Inspect `Traffic_Flow_Dataset.csv`

In [ ]:
# Load real sensor dataset
df = pd.read_csv('Traffic_Flow_Dataset.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Feature Selection & Preprocessing
We select the core numerical traffic flow features for multi-factor clustering:
- `Traffic_Volume`: Flow rate (veh/hr)
- `Vehicle_Speed_kmph`: Actual average speed (km/h)
- `Vehicle_Density`: Vehicles per km
- `Queue_Length_m`: Standstill queue distance (meters)
- `Signal_Delay_sec`: Signal control delay (seconds)
- `Hour_of_Day`: Time of day (0-23)
- `Lane_Count`: Number of road lanes

In [ ]:
feature_cols = [
    'Traffic_Volume', 
    'Vehicle_Speed_kmph', 
    'Vehicle_Density', 
    'Queue_Length_m', 
    'Signal_Delay_sec', 
    'Hour_of_Day', 
    'Lane_Count'
]

# Check for missing values
print("Missing values per column:")
print(df[feature_cols].isnull().sum())

# Correlation Heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(df[feature_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Traffic Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Z-Score Standardization (`StandardScaler`)
Normalizes values across different scales ($Z = \frac{x - \mu}{\sigma}$) so large numbers (Volume ~ 2000) don't overpower smaller numbers (Lanes ~ 4).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])
print("Scaled Feature Matrix Shape:", X_scaled.shape)

## 5. Elbow Method (WCSS Inertia) & Silhouette Score Evaluation

In [ ]:
wcss = []
silhouette_scores = []
K_range = range(2, 7)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)

# Plot Elbow Curve & Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, wcss, marker='o', color='#0284c7', linewidth=2.5)
ax1.set_title('Elbow Method (WCSS Inertia vs K)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Within-Cluster Sum of Squares')

ax2.plot(K_range, silhouette_scores, marker='s', color='#059669', linewidth=2.5)
ax2.set_title('Silhouette Score vs K', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

## 6. K-Means ML Model Fitting (K=4)
We fit K-Means with $K=4$ clusters corresponding to: `Free Flow`, `Moderate Traffic`, `Heavy Bottleneck`, `Critical Gridlock`.

In [ ]:
kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
df['ClusterID'] = kmeans_model.fit_predict(X_scaled)

# Map cluster rank by mean speed
speed_means = df.groupby('ClusterID')['Vehicle_Speed_kmph'].mean().sort_values(ascending=False)
rank_map = {old_id: new_rank for new_rank, old_id in enumerate(speed_means.index)}
df['ClusterID_Ranked'] = df['ClusterID'].map(rank_map)

cluster_names = {
    0: 'Free Flow (Smooth Traffic)',
    1: 'Moderate Traffic Flow',
    2: 'Heavy Bottleneck',
    3: 'Critical Gridlock / Severe'
}
df['ClusterName'] = df['ClusterID_Ranked'].map(cluster_names)

print("Segment Telemetry Count per Cluster:")
print(df['ClusterName'].value_counts())

# Display Cluster Averages
df.groupby('ClusterName')[feature_cols].mean().round(2)

## 7. Cluster Visualizations: Vehicle Speed vs Traffic Volume & Density

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='Vehicle_Speed_kmph', 
    y='Traffic_Volume', 
    hue='ClusterName', 
    style='ClusterName', 
    s=90, 
    palette=['#059669', '#d97706', '#ea580c', '#dc2626'],
    alpha=0.85
)
plt.title('Real Dataset Traffic Clusters: Vehicle Speed (km/h) vs Flow Volume (veh/hr)', fontsize=13, fontweight='bold')
plt.xlabel('Vehicle Speed (km/h)')
plt.ylabel('Traffic Volume (veh/hr)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Export Processed Telemetry Dataset

In [ ]:
output_file = 'Traffic_Flow_Clustered_Results.csv'
df.to_csv(output_file, index=False)
print(f"Successfully exported clustered traffic flow dataset to '{output_file}'.")